# Claude Skills 简介

了解如何使用 Claude 的 Skills 功能，借助 Excel、PowerPoint 和 PDF 生成，创建专业文档、分析数据并自动化业务工作流。

> **实际效果：**你所学习的 Skills 支撑着 Claude 的文件创建能力！查看 **[Claude Creates Files](https://www.anthropic.com/news/create-files)**，了解这些 Skills 如何让 Claude 直接在 Claude.ai 中创建和编辑文档。

## 目录

1. [环境搭建与安装](#setup)
2. [理解 Skills](#understanding)
3. [发现可用的 Skills](#discovering)
4. [快速上手：Excel](#excel-quickstart)
5. [快速上手：PowerPoint](#powerpoint-quickstart)
6. [快速上手：PDF](#pdf-quickstart)
7. [故障排查](#troubleshooting)

## 1. 环境搭建与安装 {#setup}

### 前置条件

开始前，请确认具备以下条件：
- Python 3.8 或更高版本
- 来自 [console.anthropic.com](https://console.anthropic.com/) 的 Anthropic API 密钥

### 环境配置（仅首次）

**如果尚未配置环境**，请按以下步骤操作：

#### 步骤 1：创建虚拟环境

```bash
# Navigate to the skills directory
cd /path/to/claude-cookbooks/skills

# Create virtual environment
python -m venv venv

# Activate it
source venv/bin/activate  # On macOS/Linux
# OR
venv\Scripts\activate     # On Windows
```

#### 步骤 2：安装依赖

```bash
# With venv activated, install requirements
pip install -r requirements.txt
```

#### 步骤 3：在 VSCode/Jupyter 中选择内核

**在 VSCode 中：**
1. 打开本笔记本
2. 点击右上角的内核选择器（例如 “Python 3.11.x”）
3. 选择 “Python Environments...”
4. 选择 `./venv/bin/python` 解释器

**在 Jupyter 中：**
1. 从 Kernel 菜单 → Change Kernel
2. 选择与你的 venv 对应的内核

#### 步骤 4：配置 API 密钥

```bash
# Copy the example file
cp .env.example .env

# Edit .env and add your API key:
# ANTHROPIC_API_KEY=sk-ant-api03-...
```

### 快速检查安装

运行下面的单元格，确认环境已正确配置：

**如果上方出现任何 ❌ 或 ⚠️ 警告**，请先完成配置步骤再继续。

**如果 anthropic SDK 版本过旧（需要 0.71.0 或更高版本）：**
```bash
pip install anthropic>=0.71.0
```
然后**重启 Jupyter 内核**以加载新版本。

---

### API 配置

现在加载 API 密钥并配置客户端：

### API 配置

**⚠️ 重要**：在 skills 目录下创建 `.env` 文件：

```bash
# Copy the example file
cp ../.env.example ../.env
```

然后编辑 `../.env`，加入你的 Anthropic API 密钥。

In [ ]:
import os
import sys
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))


from anthropic import Anthropic
from dotenv import load_dotenv

# Import our file utilities
from file_utils import (
    download_all_files,
    extract_file_ids,
    get_file_info,
    print_download_summary,
)

# Load environment variables from parent directory
load_dotenv(Path.cwd().parent / ".env")

API_KEY = os.getenv("ANTHROPIC_API_KEY")
MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")

if not API_KEY:
    raise ValueError(
        "ANTHROPIC_API_KEY not found. Copy ../.env.example to ../.env and add your API key."
    )

# Initialize client
# Note: We'll add beta headers per-request when using Skills
client = Anthropic(api_key=API_KEY)

# Create outputs directory if it doesn't exist
OUTPUT_DIR = Path.cwd().parent / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ API key loaded")
print(f"✓ Using model: {MODEL}")
print(f"✓ Output directory: {OUTPUT_DIR}")
print("\n📝 Note: Beta headers will be added per-request when using Skills")

### 测试连接

确认 API 连接可用：

In [ ]:
# Simple test to verify API connection
test_response = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "Say 'Connection successful!' if you can read this.",
        }
    ],
)

print("API Test Response:")
print(test_response.content[0].text)
print(
    f"\n✓ Token usage: {test_response.usage.input_tokens} in, {test_response.usage.output_tokens} out"
)

## 2. 理解 Skills {#understanding}

### 什么是 Skills？

**Skills** 是组织好的指令、可执行代码和资源包，使 Claude 具备面向特定任务的专门能力。可以把它们看作 Claude 能够动态发现和加载的“专业知识包”。

📖 阅读我们的工程博客文章 [用 Skills 让智能体适应真实世界](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills)

### 为什么 Skills 重要

在了解了 MCP（Model Context Protocol）和工具之后，你可能会想知道 Skills 为何重要：

- **Skills 比单个工具更宏观** - 它们把指令、代码和资源结合在一起
- **Skills 可组合** - 多个 Skills 无缝协同工作
- **Skills 高效** - 渐进式披露意味着只为实际使用的部分付出成本
- **Skills 包含经过验证的代码** - 可靠运行的辅助脚本，节省时间并减少错误

### 主要收益

- **专家级表现**：无需学习曲线即可获得专业结果
- **经过验证的辅助脚本**：Skills 包含经过测试、可运行的代码，Claude 可立即使用
- **组织知识**：把公司工作流和最佳实践打包起来
- **成本效益**：渐进式披露把词元用量降到最低
- **可靠性**：预先测试过的脚本意味着更少的错误和更一致的结果
- **时间节省**：Claude 使用现成方案，而不是从零生成代码
- **可组合**：多个 Skills 协同完成复杂工作流

### 渐进式披露架构

Skills 采用三层加载模型：

![Progressive Disclosure - How Skills Load](../assets/prog-disc-1.png)

1. **元数据**（name：64 字符，description：1024 字符）：Claude 看到 Skills 的名称和描述
2. **完整指令**（<5k 词元）：当该 Skills 相关时加载
3. **关联文件**：仅在需要时才加载的额外资源

![Progressive Disclosure Stages](../assets/prog-disc-2.png)

这样既保持运行高效，又能在需要时提供深度专业知识。最初，Claude 只看到 SKILL.md 的 YAML 前置元数据。只有当某个 Skills 相关时，Claude 才加载完整内容，包括其中的辅助脚本和资源。

### Skills 类型

| 类型 | 说明 | 示例 |
|------|-------------|----------|
| **Anthropic 托管** | 由 Anthropic 维护的预置 Skills | `xlsx`、`pptx`、`pdf`、`docx` |
| **自定义** | 面向特定工作流的用户自定义 Skills | 品牌规范、财务模型 |

### Skills 概念总览

![Skills Conceptual Diagram](../assets/skills-conceptual-diagram.png)

这张图说明了：
- **Skills 目录结构**：Skills 如何通过 SKILL.md 和支持文件进行组织
- **YAML 前置元数据**：Claude 最初看到的元数据
- **渐进式加载**：Skills 如何被按需发现和加载
- **可组合性**：多个 Skills 在同一次请求中协同工作

### Skills 如何与代码执行配合

Skills 需要启用**代码执行**工具。典型工作流如下：

```python
# Use client.beta.messages.create() for Skills support
response = client.beta.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=4096,
    container={
        "skills": [
            {"type": "anthropic", "skill_id": "xlsx", "version": "latest"}
        ]
    },
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[{"role": "user", "content": "Create an Excel file..."}],
    # Use betas parameter instead of extra_headers
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"]
)
```

**会发生什么：**
1. Claude 收到你的请求，并已加载 xlsx skill
2. Claude 使用代码执行创建文件
3. 响应中包含所创建文件的 `file_id`
4. 你使用 **Files API** 下载该文件

**重要：Beta API**
- 使用 `client.beta.messages.create()`（而不是 `client.messages.create()`）
- `container` 参数仅在 beta API 中可用
- 使用 `betas` 参数启用 beta 功能：
  - `code-execution-2025-08-25` - 启用代码执行
  - `files-api-2025-04-14` - 下载文件所必需
  - `skills-2025-10-02` - 启用 Skills 功能

⚠️ **说明**：使用 Skills 时，请求中必须包含 code_execution 工具。

### 词元用量优化

与在提示词中给出指令相比，Skills 大幅减少词元用量：

| 方式 | 词元成本 | 表现 |
|----------|------------|-------------|
| 手工编写指令 | 每次请求 5,000-10,000 词元 | 质量不稳定 |
| Skills（仅元数据） | 极少（仅名称/描述） | 专家级 |
| Skills（完整加载） | 使用该 Skills 时约 5,000 词元 | 专家级 |

**关键优势**：可以在提示词中放入多个 Skills 而不使其膨胀。每个 Skills 在实际使用之前只消耗元数据（名称 + 描述）。

**示例**：创建带格式的 Excel 文件
- 不使用 Skills：需要约 8,000 词元预先说明全部 Excel 功能
- 使用 Skills：初始只有极少的元数据开销，仅在调用 Excel skill 时约 5,000 词元
- **关键认识**：98% 的节省适用于初始上下文。一旦使用某个 Skills，完整指令就会被加载。

**额外收益：**
- Skills 包含已知可用的辅助脚本，提升可靠性
- Claude 使用经过验证的代码模式而不从零生成，节省时间
- 获得更一致、更专业的结果

### ⏱️ 预期生成时间

**⚠️ 重要**：使用 Skills 生成文档需要代码执行和文件创建，需要花费时间。请耐心等待，让单元格运行完成。

**观察到的生成时间：**
- **Excel 文件**：约 2 分钟（含图表和格式）
- **PowerPoint 演示文稿**：约 1-2 分钟（含图表的简单 2 页演示文稿）
- **PDF 文档**：约 40-60 秒（简单文档）

**预期情况：**
- 单元格运行期间显示 `[*]`
- 可能看到 “Executing...” 状态持续 1-2 分钟
- **不要中断单元格** - 让其完整运行结束

**💡 建议：**
1. **从简单开始**：先用最小示例验证环境
2. **逐步增加复杂度**：逐项添加功能
3. **耐心等待**：操作通常耗时 40 秒到 2 分钟
4. **说明**：非常复杂的文档可能耗时更长 - 保持示例聚焦

## 3. 发现可用的 Skills {#discovering}

### 列出全部内置 Skills

我们来查看有哪些 Anthropic 托管的 Skills 可用：

In [ ]:
# List all available Anthropic skills
# Note: Skills API requires the skills beta header
client_with_skills_beta = Anthropic(
    api_key=API_KEY, default_headers={"anthropic-beta": "skills-2025-10-02"}
)

skills_response = client_with_skills_beta.beta.skills.list(source="anthropic")

print("Available Anthropic-Managed Skills:")
print("=" * 80)

for skill in skills_response.data:
    print(f"\n📦 Skill ID: {skill.id}")
    print(f"   Title: {skill.display_title}")
    print(f"   Latest Version: {skill.latest_version}")
    print(f"   Created: {skill.created_at}")

    # Get version details
    try:
        version_info = client_with_skills_beta.beta.skills.versions.retrieve(
            skill_id=skill.id, version=skill.latest_version
        )
        print(f"   Name: {version_info.name}")
        print(f"   Description: {version_info.description}")
    except Exception as e:
        print(f"   (Unable to fetch version details: {e})")

print(f"\n\n✓ Found {len(skills_response.data)} Anthropic-managed skills")

### 理解 Skills 元数据

每个 skill 都有：
- **skill_id**：唯一标识符（例如 “xlsx”、“pptx”）
- **version**：版本号或 “latest”
- **name**：便于阅读的名称
- **description**：该 skill 的作用
- **directory**：skill 的目录结构

### 版本策略

- Anthropic 的 skills 使用 `"latest"`（推荐）
- Anthropic 会自动更新 skills
- 生产环境为求稳定应锁定具体版本
- 自定义 skills 用纪元时间戳作为版本号

### 示例：月度预算电子表格

我们从两个示例开始——一个简单的一句话请求，一个详细的请求。

#### 简单示例（1-2 行）
先用一个最小化提示词看看 Skills 如何工作：

```python
# Simple prompt - Skills handle the complexity
prompt = "Create a quarterly sales report Excel file with revenue data and a chart"
```

#### 详细示例
如需更强的控制，可以给出具体要求：
- 收入与支出类别
- 用于合计的公式
- 基本格式

### 示例：月度预算电子表格

我们创建一个简单的预算电子表格，包含：
- 收入与支出类别
- 用于合计的公式
- 基本格式

**⏱️ 说明**：Excel 生成通常需要 **1-2 分钟**（含图表和格式）。单元格运行期间显示 `[*]`——请耐心等待！

In [ ]:
# Create an Excel budget spreadsheet
excel_response = client.beta.messages.create(  # Note: Using beta.messages for Skills support
    model=MODEL,
    max_tokens=4096,
    container={"skills": [{"type": "anthropic", "skill_id": "xlsx", "version": "latest"}]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": """Create a monthly budget Excel spreadsheet with the following:

Income:
- Salary: $5,000
- Freelance: $1,200
- Investments: $300

Expenses:
- Rent: $1,500
- Utilities: $200
- Groceries: $600
- Transportation: $300
- Entertainment: $400
- Savings: $1,000

Include:
1. Formulas to calculate total income and total expenses
2. A formula for net savings (income - expenses)
3. Format currency values properly
4. Add a simple column chart showing income vs expenses
5. Use professional formatting with headers
""",
        }
    ],
    # Use betas parameter for beta features
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("Excel Response:")
print("=" * 80)
for content in excel_response.content:
    if content.type == "text":
        print(content.text)
    elif content.type == "tool_use":
        print(f"\n🔧 Tool: {content.name}")
        if hasattr(content, "input"):
            print(f"   Input preview: {str(content.input)[:200]}...")

print("\n\n📊 Token Usage:")
print(f"   Input: {excel_response.usage.input_tokens}")
print(f"   Output: {excel_response.usage.output_tokens}")

### 下载 Excel 文件

现在提取 file_id 并下载生成的 Excel 文件：

In [ ]:
# Extract file IDs from the response
file_ids = extract_file_ids(excel_response)

if file_ids:
    print(f"✓ Found {len(file_ids)} file(s)\n")

    # Download all files
    results = download_all_files(
        client, excel_response, output_dir=str(OUTPUT_DIR), prefix="budget_"
    )

    # Print summary
    print_download_summary(results)

    # Show file details
    for file_id in file_ids:
        info = get_file_info(client, file_id)
        if info:
            print("\n📄 File Details:")
            print(f"   Filename: {info['filename']}")
            print(f"   Size: {info['size'] / 1024:.1f} KB")
            print(f"   Created: {info['created_at']}")
else:
    print("❌ No files found in response")
    print("\nDebug: Response content types:")
    for i, content in enumerate(excel_response.content):
        print(f"  {i}. {content.type}")

**✨ 刚才发生了什么？**

1. Claude 使用 `xlsx` skill 创建了专业的 Excel 文件
2. 该 skill 处理了所有 Excel 特有的格式和公式
3. 文件在 Claude 的代码执行环境中创建
4. 我们从响应中提取了 `file_id`
5. 我们使用 Files API 下载了该文件
6. 文件已保存在 `outputs/budget_*.xlsx`

在 Excel 中打开该文件查看结果！

## 5. 快速上手：PowerPoint {#powerpoint-quickstart}

现在使用 `pptx` skill 创建 PowerPoint 演示文稿。

### 示例：营收演示文稿

#### 简单示例（1 行）
```python
# Minimal prompt - let Skills handle the details
prompt = "Create an executive summary presentation with 3 slides about Q3 results"
```

#### 详细示例
**说明**：这里刻意保持简单（2 页幻灯片、1 个图表），以减少生成时间并演示核心功能。

### 示例：简单的营收演示文稿

**说明**：这里刻意保持简单（2 页幻灯片、1 个图表），以减少生成时间并演示核心功能。

In [ ]:
# Create a PowerPoint presentation
pptx_response = client.beta.messages.create(
    model=MODEL,
    max_tokens=4096,
    container={"skills": [{"type": "anthropic", "skill_id": "pptx", "version": "latest"}]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": """Create a simple 2-slide PowerPoint presentation:

Slide 1: Title slide
- Title: "Q3 2025 Results"
- Subtitle: "Acme Corporation"

Slide 2: Revenue Overview
- Title: "Quarterly Revenue"
- Add a simple column chart showing:
  - Q1: $12M
  - Q2: $13M
  - Q3: $14M

Use clean, professional formatting.
""",
        }
    ],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("PowerPoint Response:")
print("=" * 80)
for content in pptx_response.content:
    if content.type == "text":
        print(content.text)

print("\n\n📊 Token Usage:")
print(f"   Input: {pptx_response.usage.input_tokens}")
print(f"   Output: {pptx_response.usage.output_tokens}")

### 下载 PowerPoint 文件

In [ ]:
# Download the PowerPoint file
file_ids = extract_file_ids(pptx_response)

if file_ids:
    results = download_all_files(
        client, pptx_response, output_dir=str(OUTPUT_DIR), prefix="q3_review_"
    )

    print_download_summary(results)

    print("\n✅ Open the presentation in PowerPoint or Google Slides to view!")
else:
    print("❌ No files found in response")

**⏱️ 说明**：简单文档的 PDF 生成通常需要 **1-2 分钟**。单元格运行期间显示 `[*]`——请耐心等待！

### 示例：PDF 文档

#### 简单示例（1 行）
```python
# Quick PDF generation
prompt = "Create a professional invoice PDF for $500 consulting services"
```

#### 详细示例：收据
**说明**：这里刻意保持简单，以确保格式整洁。

### 示例：简单收据

**说明**：这里刻意保持简单，以确保格式整洁。

In [ ]:
# Create a PDF receipt
pdf_response = client.beta.messages.create(
    model=MODEL,
    max_tokens=4096,
    container={"skills": [{"type": "anthropic", "skill_id": "pdf", "version": "latest"}]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": """Create a simple receipt PDF:

RECEIPT

Acme Corporation
Date: January 15, 2025
Receipt #: RCT-2025-001

Customer: Jane Smith

Items:
- Product A: $50.00
- Product B: $75.00
- Product C: $25.00

Subtotal: $150.00
Tax (8%): $12.00
Total: $162.00

Thank you for your business!

Use simple, clean formatting with clear sections.
""",
        }
    ],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("PDF Response:")
print("=" * 80)
for content in pdf_response.content:
    if content.type == "text":
        print(content.text)

print("\n\n📊 Token Usage:")
print(f"   Input: {pdf_response.usage.input_tokens}")
print(f"   Output: {pdf_response.usage.output_tokens}")

### 下载并校验 PDF

In [ ]:
# Download the PDF file
file_ids = extract_file_ids(pdf_response)

if file_ids:
    results = download_all_files(
        client, pdf_response, output_dir=str(OUTPUT_DIR), prefix="receipt_"
    )

    print_download_summary(results)

    # Verify PDF integrity
    for result in results:
        if result["success"]:
            file_path = result["output_path"]
            file_size = result["size"]

            # Basic PDF validation
            with open(file_path, "rb") as f:
                header = f.read(5)
                if header == b"%PDF-":
                    print(f"\n✅ PDF file is valid: {file_path}")
                    print(f"   File size: {file_size / 1024:.1f} KB")
                else:
                    print(f"\n⚠️ File may not be a valid PDF: {file_path}")
else:
    print("❌ No files found in response")

## 7. 故障排查 {#troubleshooting}

### 常见问题与解决方案

### 问题 1：未找到 API 密钥

**错误：**
```
ValueError: ANTHROPIC_API_KEY not found
```

**解决方案：**
1. 确认上级目录中存在 `.env` 文件
2. 检查是否已设置 `ANTHROPIC_API_KEY=sk-ant-api03-...`
3. 创建或编辑 `.env` 后重启 Jupyter 内核

### 问题 2：无法识别 container 参数

**错误：**
```
TypeError: Messages.create() got an unexpected keyword argument 'container'
```

**解决方案：**使用 `client.beta.messages.create()` 而不是 `client.messages.create()`：
```python
# ✅ Correct - use beta.messages
response = client.beta.messages.create(
    model=MODEL,
    container={"skills": [...]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[...],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"]
)

# ❌ Incorrect - regular messages doesn't support container
response = client.messages.create(
    model=MODEL,
    container={"skills": [...]},  # Error!
    messages=[...]
)
```

### 问题 3：Skills Beta 需要代码执行工具

**错误：**
```
BadRequestError: Skills beta requires the code_execution tool to be included in the request.
```

**解决方案：**使用 Skills 时，必须包含 code_execution 工具：
```python
# ✅ Correct
response = client.beta.messages.create(
    model=MODEL,
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[...],
    betas=["...", "skills-2025-10-02"]
)

# ❌ Incorrect - missing code_execution tool
response = client.beta.messages.create(
    model=MODEL,
    messages=[...],
    betas=["...", "skills-2025-10-02"]
)
```

### 问题 4：响应中没有找到文件

**错误：**
```
❌ No files found in response
```

**解决方案：**
1. 检查请求中是否包含代码执行工具
2. 确认该 skill 已加载（查看响应内容）
3. 确认任务确实需要创建文件
4. 在响应文本中查找错误信息

### 问题 5：文件下载失败

**错误：**
```
Error retrieving file: File not found
```

**解决方案：**
1. 文件在 Anthropic 服务器上的存续时间可能有限
2. 创建文件后立即下载
3. 检查 file_id 是否已正确从响应中提取
4. 确认 betas 列表中包含了 Files API beta

### 词元优化技巧

1. **使用 “latest” 版本**处理 Anthropic 的 skills - 会自动优化
2. **批量操作** - 尽可能在一次对话中创建多个文件
3. **复用容器** - 使用先前响应中的 `container.id`，避免重新加载 skills
4. **表述明确** - 清晰的指令意味着更少的迭代

### API 速率限制

如果遇到速率限制：
- 为重试实现指数退避
- 对多个文件使用批次处理
- 考虑升级 API 层级以获得更高限额

## 后续步骤

🎉 **恭喜！**你已经掌握了 Claude Skills 的基础知识。

### 观看 Skills 的实际应用

查看官方公告，了解这些 Skills 如何支撑 Claude 的文件创建能力：
- **[Claude Creates Files](https://www.anthropic.com/news/create-files)** - 了解 Skills 如何让 Claude 直接创建和编辑 Excel、PowerPoint 和 PDF 文件

### 继续学习：

- **[笔记本 2：金融应用](02_skills_financial_applications.ipynb)** - 使用金融数据的真实业务用例
- **[笔记本 3：自定义 Skills 开发](03_skills_custom_development.ipynb)** - 构建你自己的专门 skills

### 支持文章：

- 📚 **[用 Skills 让 Claude 学会你的工作方式](https://support.claude.com/en/articles/12580051-teach-claude-your-way-of-working-using-skills)** - 使用 Skills 的用户指南
- 🛠️ **[如何通过对话与 Claude 一起创建 skill](https://support.claude.com/en/articles/12599426-how-to-create-a-skill-with-claude-through-conversation)** - 交互式 skill 创建指南

### 资源：

- [Claude API 文档](https://docs.anthropic.com/en/api/messages)
- [Skills 文档](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/overview)
- [Skills 最佳实践](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/best-practices)
- [Files API 文档](https://docs.claude.com/en/api/files-content)
- [Claude 支持](https://support.claude.com)

### 试试这些实验：

1. 从简单的一句话提示词开始，观察 Skills 的实际效果
2. 修改预算示例，加入更多类别
3. 用你自己的数据创建演示文稿
4. 生成结合文本与表格的 PDF 报告
5. 在同一次请求中组合使用多个 skills